<a href="https://colab.research.google.com/github/amitroy1423/LangChain-Learning-GenAI/blob/main/13.RAG/rag_using_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

YouTube RAG — Chat with YouTube Videos

**Problem Statement**

Long YouTube videos such as lectures, podcasts, and tutorials can contain a large amount of information. Finding a specific answer or summarizing a particular topic requires watching a significant portion of the video. This makes information retrieval time-consuming.

**Solution**

Build a **Retrieval-Augmented Generation (RAG)** system that allows users to ask questions about a YouTube video's content and receive answers based only on its transcript.

**RAG Workflow**

1. **Indexing** — Fetch the YouTube transcript and divide it into smaller chunks.
2. **Embedding** — Convert the transcript chunks into numerical vectors using the Gemini embedding model.
3. **Vector Store** — Store the embeddings in a FAISS vector store for efficient semantic search.
4. **Retrieval** — When the user asks a question, retrieve the most relevant transcript chunks.
5. **Augmentation** — Combine the retrieved chunks with the user's question to create a context-rich prompt.
6. **Generation** — Send the prompt to the Gemini LLM to generate an answer based only on the retrieved transcript context.

**Overall Flow**

YouTube Video → Transcript → Chunks → Gemini Embeddings → FAISS → Retrieval → Context + Question → Gemini LLM → Answer

**Install Required Libraries**

This cell installs the libraries required for the YouTube RAG project.

We use:
- `youtube-transcript-api` to retrieve YouTube transcripts.
- `langchain-text-splitters` for splitting the transcript.
- `langchain-google-genai` for Gemini embeddings.
- `langchain-community` and `faiss-cpu` for the FAISS vector store.

In [1]:
!pip install -q -U youtube-transcript-api langchain-google-genai langchain-community langchain-text-splitters faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 12.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth=

**Gemini API Key Setup**

This cell loads the Gemini API key securely from Google Colab Secrets.

The key is required because Gemini will be used as the embedding model.

In [3]:
from google.colab import userdata
import os

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

# **Step 1a — Indexing: Document Ingestion / Transcript Loading**

**1. Load YouTube Transcript**

This cell retrieves the transcript of a YouTube video using the YouTube Transcript API.

The video ID is used instead of the complete YouTube URL.

The transcript initially consists of timestamped snippets. These snippets are combined into one continuous text string so that the transcript can later be split into smaller chunks.

In [14]:
from youtube_transcript_api import YouTubeTranscriptApi

video_id = "9QXCkMTbrSk"

ytt_api = YouTubeTranscriptApi()

try:
    fetched_transcript = ytt_api.fetch(
        video_id,
        languages=["en"]
    )

    transcript_list = fetched_transcript.to_raw_data()

    transcript = " ".join(
        chunk["text"] for chunk in transcript_list
    )

    print(transcript)

except Exception as e:
    print("Could not fetch transcript:", e)

France invented a lot of early technologies in the world. But today if you look the scenario, the largest technology companies in the world are either American or Chinese. Many of them are led by Indian origin CEOs. Where did France lack? >> I I I think the missing point is first scale, not just for the French but for all the Europeans. Second, capital. And the third point, we have to find more appetite for risk. France has been an all-weather friend for India. The question is no longer whether India innovates. The question is who will innovate with India? And France is the unique clear answer. Jai Ho. Do you think this is going to be an example for the world that now the new world order is about multipolarity? Do you think India and France are setting an example? >> French President Emmanuel Macron is the chief guest. >> Emmanuel Macron has been invited as the chief guest at Republic Day parade. We have developed during the last few years a series of cooperation and partnerships in di

**2. View Transcript Data**

This cell displays the raw transcript snippets returned by the YouTube Transcript API.

Each item contains information such as the text, start time, and duration.

In [15]:
transcript_list

[{'text': 'France invented a lot of early', 'start': 0.0, 'duration': 5.8},
 {'text': 'technologies in the world.', 'start': 2.68, 'duration': 3.12},
 {'text': 'But today if you look the scenario, the',
  'start': 10.0,
  'duration': 3.84},
 {'text': 'largest technology companies in the',
  'start': 11.8,
  'duration': 5.32},
 {'text': 'world are either American or Chinese.',
  'start': 13.84,
  'duration': 5.2},
 {'text': 'Many of them are led by Indian origin',
  'start': 17.12,
  'duration': 3.68},
 {'text': 'CEOs.', 'start': 19.04, 'duration': 3.64},
 {'text': 'Where did France lack?', 'start': 20.8, 'duration': 4.0},
 {'text': '>> I I I think the missing point is first',
  'start': 22.68,
  'duration': 3.96},
 {'text': 'scale, not just for the French but for',
  'start': 24.8,
  'duration': 4.76},
 {'text': 'all the Europeans. Second, capital. And',
  'start': 26.64,
  'duration': 5.36},
 {'text': 'the third point, we have to find more',
  'start': 29.56,
  'duration': 5.88},
 {'t

# **Step 1b — Indexing: Text Splitting**

**3. Split Transcript into Chunks**

YouTube transcripts can be very long, so sending the complete transcript to an LLM is inefficient.

This cell uses `RecursiveCharacterTextSplitter` to divide the transcript into smaller chunks.

`chunk_size=1000` specifies the target size of each chunk.

`chunk_overlap=200` keeps some text from the previous chunk in the next chunk so that important context is not lost at chunk boundaries.

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.create_documents([transcript])

**4. Check Number of Chunks and Inspect a Chunk**

This cell counts how many chunks were created from the complete YouTube transcript. Then displays one chunk from the transcript.

Each chunk is a LangChain `Document` containing `page_content` and `metadata`.

In [17]:
len(chunks)

44

In [19]:
chunks[43]

Document(metadata={}, page_content="can improve and keep doing that better, and what all did we do wrong so that we never repeat it. And at the same time, please give us suggestions of who's the next guest that you want to see on the podcast. And don't forget to share this episode with at least one person who will get some insights because one conversation is enough to give people enough ideas to change their lives. I'll see you next time. Until then, keep figuring out. And also, don't forget to subscribe the channel. >> [music] [music]")

# **Step 1c & 1d — Embedding Generation and Vector Store**

**5. Create Gemini Embedding Model**

This cell initializes the Gemini embedding model.

The embedding model converts each transcript chunk into a numerical vector so that semantically similar pieces of text can be found later during retrieval.

We use `gemini-embedding-001` with 32 dimensions for this learning project.

In [20]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    output_dimensionality=32
)

**6. Create FAISS Vector Store**

This cell converts the transcript chunks into embeddings and stores them in a FAISS vector store.

FAISS allows us to efficiently search for transcript chunks that are semantically similar to a user's question.

In [21]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    chunks,
    embeddings
)

**7. View FAISS Document IDs**

This cell displays the mapping between FAISS's internal vector positions and the document IDs stored in the LangChain document store.

The IDs can be used to identify individual documents stored in the vector store.

In [22]:
vector_store.index_to_docstore_id

{0: '881aeb67-c805-4869-b3f4-759d1a48c54d',
 1: '4e506b75-548d-4c0f-8fe8-915f61eb32fe',
 2: '0dcee0f6-5c15-4a7e-9f60-4eae8ec298cc',
 3: '4f93574c-2bc9-4bcc-90c5-3a0e8621ab05',
 4: '1c0b5294-e030-489f-8f90-614ab03ec89d',
 5: '5486298a-4173-4ef7-95d8-979b248b5673',
 6: 'dde70b8d-fbb2-4eb6-a880-303fd6e38991',
 7: 'a846dbc9-7915-4ad0-bd25-00136f53fff5',
 8: 'd4bda4f4-07e1-4380-bfae-7ebaf0974126',
 9: 'd8158768-e2e6-46d3-837d-2e626a5b545e',
 10: '7ae0b030-fee1-486d-b56c-b841949e92fa',
 11: '38dbfc36-74e6-413c-bf1e-7ef112531993',
 12: 'a5308525-618e-4dba-9852-546228ad0932',
 13: '2d42d771-cbbd-4936-8755-5d27aebc2a66',
 14: 'c0aa3cf4-9f0e-4025-bb86-4982b237686b',
 15: '7e3d4c70-5954-424a-8102-7039b98bc1c0',
 16: 'd4f8caf3-320b-4c05-b5cc-ea78e95b5462',
 17: 'e7138491-5bb7-4647-9aa2-f0223355184a',
 18: '003b6c26-cfe7-4452-b385-46aadcecc6ce',
 19: 'ca7836da-b727-4e95-8360-468fe6f217d9',
 20: '1c73530c-17dc-4f7f-aa14-803b8a035653',
 21: 'fcab759b-80d3-46c5-a268-9e3b002fea95',
 22: 'ac920cc8-bc77-

**8. Retrieve One Stored Document**

This cell retrieves one document from the FAISS-backed document store using its document ID.

The ID should come from the `index_to_docstore_id` output rather than being copied from the instructor's notebook.

In [23]:
first_doc_id = list(vector_store.index_to_docstore_id.values())[0]

print("Document ID:", first_doc_id)

result = vector_store.get_by_ids([first_doc_id])

print(result)

Document ID: 881aeb67-c805-4869-b3f4-759d1a48c54d
[Document(id='881aeb67-c805-4869-b3f4-759d1a48c54d', metadata={}, page_content='France invented a lot of early technologies in the world. But today if you look the scenario, the largest technology companies in the world are either American or Chinese. Many of them are led by Indian origin CEOs. Where did France lack? >> I I I think the missing point is first scale, not just for the French but for all the Europeans. Second, capital. And the third point, we have to find more appetite for risk. France has been an all-weather friend for India. The question is no longer whether India innovates. The question is who will innovate with India? And France is the unique clear answer. Jai Ho. Do you think this is going to be an example for the world that now the new world order is about multipolarity? Do you think India and France are setting an example? >> French President Emmanuel Macron is the chief guest. >> Emmanuel Macron has been invited as 

# **Step 2 — Retrieval**

**1. Create a Retriever**

This cell converts the FAISS vector store into a retriever.

The retriever will search the stored transcript chunks and return the most relevant chunks for a user's query.

`search_type="similarity"` performs similarity-based retrieval.

`k=4` means the retriever will return the 4 most relevant transcript chunks.

In [24]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

**2. View the Retriever**

This cell displays the retriever object and its configuration.

In [25]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7a38586f2270>, search_kwargs={'k': 4})

**3. Ask a Question About the Video**

This cell sends a question to the retriever.

The question should be related to the content of the YouTube video.

The retriever searches the transcript chunks stored in FAISS and returns the most relevant chunks.

In [26]:
query = "What are the three main challenges France faces in trying to build a sovereign AI ecosystem compared to the US and China?"

results = retriever.invoke(query)

results

[Document(id='ac920cc8-bc77-4217-adae-c0ec21dea6fa', metadata={}, page_content="in India, you you you you you train every year more engineers than in US and Europe together. Yeah. So, it's I mean, it's such a resource. It's unique. True. And it means a lot of talents, a lot of people um and on top of that, you have you leapfrog in terms of education and competences. And you you have the same appetite of an big emerging countries. And you you you were right to mention payment system and UPI. And I'm obsessed by that because by the way, a lot of people are are super dependent on US solutions on payment. Uh I mean, it happens that in France, we we do have our own solution as well with the the Carte Bleue system, but um it's uh UPI reached such a scale in such a few years that it's very impressive. It's very hard to say what the moonshot will be because you never know. And especially as a president, it's very hard to say it will be this one as a moonshot because it's not a way innovation f

**4. Display Retrieved Chunks**

This cell prints the transcript chunks returned by the retriever.

These are the pieces of the video transcript that are most relevant to the question.

In [27]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
in India, you you you you you train every year more engineers than in US and Europe together. Yeah. So, it's I mean, it's such a resource. It's unique. True. And it means a lot of talents, a lot of people um and on top of that, you have you leapfrog in terms of education and competences. And you you have the same appetite of an big emerging countries. And you you you were right to mention payment system and UPI. And I'm obsessed by that because by the way, a lot of people are are super dependent on US solutions on payment. Uh I mean, it happens that in France, we we do have our own solution as well with the the Carte Bleue system, but um it's uh UPI reached such a scale in such a few years that it's very impressive. It's very hard to say what the moonshot will be because you never know. And especially as a president, it's very hard to say it will be this one as a moonshot because it's not a way innovation function. You will probably lose several times and the one you 

# **Step 3 — Augmentation**

**1. Initialize the Gemini LLM**

This cell initializes the Gemini chat model that will later use the retrieved transcript context to answer the user's question.

The model is used only for language generation. The transcript retrieval is already handled by the FAISS retriever.

In [28]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.2
)

**2. Create the Prompt Template**

This cell creates a prompt that contains two inputs:

- `context` → relevant transcript chunks retrieved from the video
- `question` → the user's question

The prompt instructs the LLM to answer only using the provided transcript context. If the context does not contain enough information, the model should say that it does not know.

In [29]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""
You are a helpful assistant.
Answer ONLY from the provided transcript context.
If the context is insufficient, just say you don't know.

{context}

Question: {question}
""",
    input_variables=["context", "question"]
)

**3. Retrieve Relevant Transcript Chunks**

This cell sends the user's question to the retriever.

The retriever searches the FAISS vector store and returns the most relevant transcript chunks.

In [35]:
question = "How does the partnership between France and India serve as a model for 'strategic autonomy' in a world dominated by two major powers?"

retrieved_docs = retriever.invoke(question)

retrieved_docs

[Document(id='ed9a11b7-8629-4f83-b848-98f16fb63daa', metadata={}, page_content="this is a geopolitical umbrella and a and a strong alliance which will always deliver. I mean, all this approach collapsed. Yeah. And and not to say that we want to be confrontational with the US. I mean, the more intense our relation could be, the more happy I will be. But having an ally, having a friend doesn't mean that you're dependent on your friend. True. And and I I really believe that when when a lot of Europeans see that this approach is adopted by a country like India, it gives them more more trust. And as they accept to be probably a little bit bolder. So, I I really believe on on the potential of this relation as well because it's it will increase what we want to do together, meaning being more independent. And once once again, what do we want to be more independent? First, because we are obsessed by this sovereign approach. What does it mean sovereign? Our fellow citizen decide for themselves. 

**4. Create the Context**

This cell combines the page content of all retrieved transcript chunks into one text string.

This combined text becomes the context that will be provided to the LLM.

In [36]:
context_text = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)

context_text

"this is a geopolitical umbrella and a and a strong alliance which will always deliver. I mean, all this approach collapsed. Yeah. And and not to say that we want to be confrontational with the US. I mean, the more intense our relation could be, the more happy I will be. But having an ally, having a friend doesn't mean that you're dependent on your friend. True. And and I I really believe that when when a lot of Europeans see that this approach is adopted by a country like India, it gives them more more trust. And as they accept to be probably a little bit bolder. So, I I really believe on on the potential of this relation as well because it's it will increase what we want to do together, meaning being more independent. And once once again, what do we want to be more independent? First, because we are obsessed by this sovereign approach. What does it mean sovereign? Our fellow citizen decide for themselves. If you are dependent technologically or regarding defense issues, you cannot\n\

**5. Add Context and Question to the Prompt**

This cell fills the prompt template with the retrieved transcript context and the user's question.

The result is the final prompt that will be passed to the Gemini model in the Generation step.

In [37]:
final_prompt = prompt.invoke({
    "context": context_text,
    "question": question
})

final_prompt

StringPromptValue(text="\nYou are a helpful assistant.\nAnswer ONLY from the provided transcript context.\nIf the context is insufficient, just say you don't know.\n\nthis is a geopolitical umbrella and a and a strong alliance which will always deliver. I mean, all this approach collapsed. Yeah. And and not to say that we want to be confrontational with the US. I mean, the more intense our relation could be, the more happy I will be. But having an ally, having a friend doesn't mean that you're dependent on your friend. True. And and I I really believe that when when a lot of Europeans see that this approach is adopted by a country like India, it gives them more more trust. And as they accept to be probably a little bit bolder. So, I I really believe on on the potential of this relation as well because it's it will increase what we want to do together, meaning being more independent. And once once again, what do we want to be more independent? First, because we are obsessed by this sove

# **Step 4 — Generation**

**1. Generate the Final Answer**

This cell sends the augmented prompt, which contains the relevant transcript context and the user's question, to the Gemini language model.

The LLM uses the provided context to generate the final answer.

In [39]:
answer = llm.invoke(final_prompt)

print(answer.text)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the provided transcript, the partnership serves as a "third way" and a model for strategic autonomy—meaning not being confrontational with the US or China, not being dependent on them, and not choosing sides. Instead, it allows countries to build their own future in full respect, driven by a sovereign approach where citizens decide for themselves. Additionally, when Europeans see this approach adopted by a country like India, it gives them more trust, makes them bolder, and helps them realize that strategic autonomy (rather than choosing one side) is the way forward.


# **Building a RAG Chain**

**1. Import Runnable Components**

This cell imports the LangChain Runnable components used to connect the retrieval and generation steps into a single RAG chain.

- `RunnableParallel` runs multiple operations using the same input.
- `RunnablePassthrough` passes the original input without changing it.
- `RunnableLambda` allows us to use a custom Python function inside the chain.
- `StrOutputParser` converts the LLM output into a plain string.

In [40]:
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
    RunnableLambda
)

from langchain_core.output_parsers import StrOutputParser

**2. Create a Function to Format Retrieved Documents**

The retriever returns a list of LangChain `Document` objects.

This function extracts the `page_content` from each document and combines them into one text string.

The resulting text will be used as the context for the LLM.

In [41]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )
    return context_text

**3. Create the Parallel Chain**

This cell creates a `RunnableParallel` chain.

The same user question is sent to two branches at the same time:

1. The `context` branch sends the question to the retriever and formats the retrieved documents.
2. The `question` branch passes the original question unchanged.

The result is a dictionary containing both the retrieved context and the original question.

In [42]:
parallel_chain = RunnableParallel({
    "context": retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough()
})

**4. Test the Parallel Chain**

This cell tests the parallel chain by passing a question related to the YouTube video.

The output should contain both the retrieved context and the original question.

In [43]:
parallel_chain.invoke("Why does President Macron believe that failing early and taking risks is essential for France to lead in 'moonshot' technologies like quantum computing?")

{'context': "we have a unique mathematicians, new capacities and a lot of very performing startups and labs. So I I really believe that in this field we can we can deliver and and have a moonshot. But but more more than that, I I just want to convey a very simple message. When we speak about innovation and moonshot, it's it's all about taking risk. And it's all about accepting to lose. True. And it seems to be super counterintuitive. >> [snorts] >> But if you want to think and deliver big, you have to take a lot of risk. Except by chance, you never succeed the first time. So if you don't accept to to fail and to start again, you will never succeed. And for me, the question for innovators, but it's true for countries and so on, is not to have people being super successful the first time. It's doesn't exist. But it's to make sure that we have people ready to to lose eight time but try nine times. Yeah. [snorts] You know, moving away from technology because you said it right, it's it's a 

**5. Create the Output Parser**

This cell creates a `StrOutputParser` that converts the model's response into a simple Python string.

In [44]:
parser = StrOutputParser()

**6. Build the Complete RAG Chain**

This cell connects the complete RAG workflow using the LangChain pipe operator.

The pipeline performs the following operations:

1. Retrieve relevant transcript chunks.
2. Format those chunks into context.
3. Keep the original question.
4. Insert the context and question into the prompt.
5. Send the prompt to the Gemini LLM.
6. Convert the LLM response into a string.

In [45]:
main_chain = parallel_chain | prompt | llm | parser

**7. Run the Complete RAG Chain**

This cell sends a question to the complete RAG chain.

The chain retrieves relevant information from the YouTube transcript and uses that context to generate the final answer with Gemini.

In [46]:
result = main_chain.invoke(
    "Can you summarize the video?"
)

print(result)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the provided transcript, the video features a podcast recorded in Mumbai with the French President (President Macron) as a guest. The discussion covers several topics, including:

* **Innovation and Risk-Taking:** The importance of taking risks, accepting failure, and trying repeatedly to succeed in fields like innovation, tech, and moonshot projects.
* **Global Order and India:** Discussing India's role, perspective, and what the world needs to understand about it as the world heads toward multipolarity.
* **Research and Entrepreneurship:** Mentioning that "entrepreneur" is a French word, and highlighting Europe's (and the UK's) active ecosystems in innovation, startups, AI, quantum technology, and mathematics, while also noting that the tech sector is largely driven by the US and China due to a question of scale. 

The transcript also includes lighter moments, such as casual greetings and a conversation about the French language with a host living in Mumbai.
